# 🛠️ Kaggle: Huấn luyện Mô hình Tùy Biến

In [ ]:
# 1. KÉO CODE TỪ GITHUB
!git clone https://github.com/Taikhoan/CT282-RIFE.git RIFE-Project
%cd /kaggle/working/RIFE-Project/

In [ ]:
# 2. NẠP DỮ LIỆU TỪ HUGGING FACE BẰNG LOAD_DATASET
!pip install -q datasets

from datasets import load_dataset
import os

DATA_DIR = "./vimeo_triplet"

if not os.path.exists(f"{DATA_DIR}/sequences"):
    print("⏳ 1. Đang tải dataset bijinc/vimeo-90k-mini từ Hugging Face...")
    ds = load_dataset("bijinc/vimeo-90k-mini")

    print(f"⏳ 2. Đang tạo cấu trúc ảnh chuẩn cho RIFE...")
    os.makedirs(f"{DATA_DIR}/sequences", exist_ok=True)

    train_list = []
    test_list = []

    for split_name in ds.keys():
        print(f"   -> Đang xử lý tập {split_name} ({len(ds[split_name])} mẫu)...")
        for i, item in enumerate(ds[split_name]):
            folder = f"{split_name}/{i:05d}"
            os.makedirs(f"{DATA_DIR}/sequences/{folder}", exist_ok=True)
            
            item['im1'].save(f"{DATA_DIR}/sequences/{folder}/im1.png")
            item['im2'].save(f"{DATA_DIR}/sequences/{folder}/im2.png")
            item['im3'].save(f"{DATA_DIR}/sequences/{folder}/im3.png")
            
            if split_name == 'train':
                train_list.append(folder)
            else:
                test_list.append(folder)

    with open(f"{DATA_DIR}/tri_trainlist.txt", "w") as f:
        f.write("\n".join(train_list))

    with open(f"{DATA_DIR}/tri_testlist.txt", "w") as f:
        f.write("\n".join(test_list if test_list else train_list[:100]))

print("✅ ĐÃ NẠP DỮ LIỆU XONG 100%! Sẵn sàng để train.")

In [ ]:
# 3. CẤU HÌNH VÀ BẮT ĐẦU HUẤN LUYỆN
ACTIVATION = 'smooth_prelu'
ATTENTION = 'eca'
SAVE_DIR = f"/kaggle/working/RIFE-Project/trained_model/modify_{ATTENTION}_{ACTIVATION}"

!python train.py \
    --model_type modify \
    --act {ACTIVATION} \
    --attn {ATTENTION} \
    --batch_size 32 \
    --epoch 40 \
    --save_dir {SAVE_DIR}

In [ ]:
#@title 📤 4. TỰ ĐỘNG COMMIT & PUSH KẾT QUẢ VỀ GITHUB
import getpass

GITHUB_USERNAME = "Ten_Github_Cua_Ban" #@param {type:"string"}
GITHUB_EMAIL = "email_cua_ban@gmail.com" #@param {type:"string"}
REPO_NAME = "CT282-RIFE" #@param {type:"string"}

# Nhập token an toàn qua hộp thoại (không bị lộ trên code GitHub)
GITHUB_TOKEN = getpass.getpass("Nhập GitHub Token (sẽ bị ẩn): ")

!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "{GITHUB_USERNAME}"

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# Chỉ thêm thư mục trained_model, không thêm dataset
!git add trained_model/
!git commit -m "Lưu kết quả train từ Kaggle [Auto Sync]"
!git push {REPO_URL} main

print("✅ ĐÃ PUSH THÀNH CÔNG KẾT QUẢ LÊN GITHUB! Trên máy tính chỉ cần gõ 'git pull'.")